# SiT-S/8 Only × CheXpert (Colab)

Trains **SiT-S/8** on CheXpert chest X-rays using **only the diffusion loss** — χωρίς REPA alignment, χωρίς encoder.

- `--enc-type None` → δεν φορτώνεται κανένας visual encoder
- `--proj-coeff 0.0` → total loss = μόνο denoising loss
- **2 κλάσεις**: 0 = Normal, 1 = Abnormal

---

### Επιλογή VAE

Σε κάθε βήμα (Prepare / Train / Generate) υπάρχουν **2 cells** — τρέξε μόνο το ένα:

| | Standard VAE | MedVAE |
|---|---|---|
| **VAE** | `stabilityai/sd-vae-ft-mse` | MedVAE (medical) |
| **Πότε** | Γρήγορο baseline | Καλύτερη ποιότητα για X-rays |

## 1. Install dependencies

In [ ]:
!pip install -q accelerate diffusers timm einops pandas medvae kaggle

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/sit-chexpert-results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Drive mounted. Results will be saved to:', RESULTS_DIR)

## 3. Setup Kaggle & download CheXpert

In [ ]:
import os, shutil, subprocess

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('/content/drive/MyDrive/kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle credentials set.')

if not os.path.isfile('/content/chexpert/train.csv'):
    print('Downloading CheXpert...')
    subprocess.run(['kaggle', 'datasets', 'download', 'ashery/chexpert',
                    '-p', '/content/chexpert', '--unzip'], check=True)
else:
    print('CheXpert already downloaded.')

print('Done.')

## 4. Clone REPA fork

In [ ]:
%%bash
if [ ! -d "/content/REPA" ]; then
    git clone https://github.com/nikiboura/REPA.git /content/REPA
fi
cd /content/REPA && git pull
echo "REPA ready. Commit: $(git rev-parse --short HEAD)"

## 5. Prepare CheXpert

Encode τις X-ray εικόνες με το επιλεγμένο VAE και αποθήκευση latents.  
**Τρέξε μόνο το ένα από τα 2 cells παρακάτω.**

In [ ]:
# ▶ OPTION A — Standard VAE (sd-vae-ft-mse)
import subprocess, sys

cmd = [
    sys.executable, '/content/REPA/dataset_preparation_scripts/prepare_chexpert.py',
    '--out-dir',       '/content/data/chexpert_256',
    '--chexpert-root', '/content/chexpert',
    '--resolution',    '256',
    '--max-samples',   '80000',
    '--vae-type',      'sd',
]
result = subprocess.run(cmd, text=True)
print('Exit code:', result.returncode)

In [ ]:
# ▶ OPTION B — MedVAE
import subprocess, sys

cmd = [
    sys.executable, '/content/REPA/dataset_preparation_scripts/prepare_chexpert.py',
    '--out-dir',       '/content/data/chexpert_256',
    '--chexpert-root', '/content/chexpert',
    '--resolution',    '256',
    '--max-samples',   '80000',
    '--vae-type',      'medvae',
]
result = subprocess.run(cmd, text=True)
print('Exit code:', result.returncode)

## 6. Train — SiT-S/8 μόνο diffusion loss

**Τρέξε μόνο το ένα από τα 2 cells παρακάτω** (ανάλογα με το VAE που έτρεξες στο βήμα 5).

In [ ]:
# ▶ OPTION A — Standard VAE
import subprocess, os, re
from tqdm.notebook import tqdm

cmd = [
    'accelerate', 'launch',
    '--mixed_precision', 'fp16',
    '--num_processes', '1',
    '/content/REPA/train.py',
    '--exp-name',            'sit-s8-chexpert-sdvae',
    '--output-dir',          RESULTS_DIR,
    '--report-to',           'none',
    '--model',               'SiT-S/8',
    '--num-classes',         '2',
    '--encoder-depth',       '8',
    '--enc-type',            'None',
    '--proj-coeff',          '0.0',
    '--data-dir',            '/content/data/chexpert_256',
    '--resolution',          '256',
    '--batch-size',          '32',
    '--num-workers',         '2',
    '--epochs',              '200',
    '--learning-rate',       '1e-4',
    '--mixed-precision',     'fp16',
    '--cfg-prob',            '0.1',
    '--path-type',           'linear',
    '--max-train-steps',     '400000',
    '--checkpointing-steps', '400000',
    '--sampling-steps',      '99999999',
]

process = subprocess.Popen(
    cmd, cwd='/content/REPA',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

pat = r'Steps:\s*(\d+)/\d+.*?loss=([\d.]+)'
pbar = tqdm(total=400000, desc='Training SiT-S/8 [SD VAE]')
last_step = 0
for line in process.stdout:
    m = re.search(pat, line)
    if m:
        step = int(m.group(1))
        if step > last_step:
            pbar.update(step - last_step)
            pbar.set_postfix({'loss': m.group(2)})
            last_step = step
    else:
        stripped = line.strip()
        if stripped:
            print(stripped, flush=True)
pbar.close()
process.wait()
print('Training complete. Checkpoint saved to Drive.')

In [ ]:
# ▶ OPTION B — MedVAE
import subprocess, os, re
from tqdm.notebook import tqdm

cmd = [
    'accelerate', 'launch',
    '--mixed_precision', 'fp16',
    '--num_processes', '1',
    '/content/REPA/train.py',
    '--exp-name',            'sit-s8-chexpert-medvae',
    '--output-dir',          RESULTS_DIR,
    '--report-to',           'none',
    '--model',               'SiT-S/8',
    '--num-classes',         '2',
    '--encoder-depth',       '8',
    '--enc-type',            'None',
    '--proj-coeff',          '0.0',
    '--data-dir',            '/content/data/chexpert_256',
    '--resolution',          '256',
    '--batch-size',          '32',
    '--num-workers',         '2',
    '--epochs',              '200',
    '--learning-rate',       '1e-4',
    '--mixed-precision',     'fp16',
    '--cfg-prob',            '0.1',
    '--path-type',           'linear',
    '--max-train-steps',     '400000',
    '--checkpointing-steps', '400000',
    '--sampling-steps',      '99999999',
    '--vae-type',            'medvae',
]

process = subprocess.Popen(
    cmd, cwd='/content/REPA',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

pat = r'Steps:\s*(\d+)/\d+.*?loss=([\d.]+)'
pbar = tqdm(total=400000, desc='Training SiT-S/8 [MedVAE]')
last_step = 0
for line in process.stdout:
    m = re.search(pat, line)
    if m:
        step = int(m.group(1))
        if step > last_step:
            pbar.update(step - last_step)
            pbar.set_postfix({'loss': m.group(2)})
            last_step = step
    else:
        stripped = line.strip()
        if stripped:
            print(stripped, flush=True)
pbar.close()
process.wait()
print('Training complete. Checkpoint saved to Drive.')

## 7. Generate samples

**Τρέξε μόνο το ένα από τα 2 cells παρακάτω** (ανάλογα με το VAE που χρησιμοποίησες στο training).

In [ ]:
# ▶ OPTION A — Standard VAE
import subprocess, os, glob

ckpts = sorted(glob.glob(f'{RESULTS_DIR}/sit-s8-chexpert-sdvae/checkpoints/*.pt'))
print('Using checkpoint:', ckpts[-1])

process = subprocess.Popen([
    'torchrun', '--nproc_per_node=1', '--master_port=29501',
    '/content/REPA/generate.py',
    '--model',                'SiT-S/8',
    '--ckpt',                 ckpts[-1],
    '--encoder-depth',        '8',
    '--num-classes',          '2',
    '--projector-embed-dims', '768',
    '--per-proc-batch-size',  '16',
    '--num-fid-samples',      '2000',
    '--path-type',            'linear',
    '--mode',                 'ode',
    '--num-steps',            '50',
    '--cfg-scale',            '1.0',
    '--resolution',           '256',
    '--vae',                  'mse',
], cwd='/content/REPA', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
print('Exit code:', process.returncode)

In [ ]:
# ▶ OPTION B — MedVAE
import subprocess, os, glob

ckpts = sorted(glob.glob(f'{RESULTS_DIR}/sit-s8-chexpert-medvae/checkpoints/*.pt'))
print('Using checkpoint:', ckpts[-1])

process = subprocess.Popen([
    'torchrun', '--nproc_per_node=1', '--master_port=29501',
    '/content/REPA/generate.py',
    '--model',                'SiT-S/8',
    '--ckpt',                 ckpts[-1],
    '--encoder-depth',        '8',
    '--num-classes',          '2',
    '--projector-embed-dims', '768',
    '--per-proc-batch-size',  '16',
    '--num-fid-samples',      '2000',
    '--path-type',            'linear',
    '--mode',                 'ode',
    '--num-steps',            '50',
    '--cfg-scale',            '1.0',
    '--resolution',           '256',
    '--vae',                  'medvae',
], cwd='/content/REPA', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
print('Exit code:', process.returncode)

## 8. Show generated images

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob

npz_files = sorted(glob.glob('/content/REPA/samples/**/*.npz', recursive=True))
data = np.load(npz_files[-1])
imgs = data[data.files[0]]
print(f'Total generated: {len(imgs)}')

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i].astype('uint8'), cmap='gray')
    ax.axis('off')
plt.suptitle('SiT-S/8 (no REPA) — CheXpert samples', fontsize=13)
plt.tight_layout()
plt.show()
print(f'From: {npz_files[-1]}')

## 9. Compute FID score

In [ ]:
import subprocess, glob, numpy as np, os
from PIL import Image
from tqdm import tqdm

subprocess.run(['pip', 'install', '-q', 'torch-fidelity'], check=True)

gen_npz = sorted(glob.glob('/content/REPA/samples/**/*.npz', recursive=True))[-1]
print('Generated:', gen_npz)

gen_dir = '/content/gen_images'
os.makedirs(gen_dir, exist_ok=True)
data = np.load(gen_npz)
imgs = data[data.files[0]]
for i, img in enumerate(tqdm(imgs, desc='Saving generated images')):
    Image.fromarray(img.astype('uint8')).save(f'{gen_dir}/{i:05d}.png')

real_dir = '/content/data/chexpert_256/images/train'

result = subprocess.run([
    'fidelity', '--gpu', '0', '--fid',
    '--input1', gen_dir,
    '--input2', real_dir,
], capture_output=True, text=True)

print(result.stdout)
print(result.stderr)